# 91. Reprocesar registros de la cuarentena (reto opcional)

Cada registro de `silver.cuarentena` guarda la fuente, las reglas que fallaron, el lote, el archivo, las fechas de ingesta y de procesamiento, y el registro original en JSON. Este notebook toma los registros de una fuente, les aplica las correcciones que se indiquen y los escribe como un archivo nuevo en la carpeta de landing de esa fuente. En la siguiente corrida el archivo entra como un lote más (`<fuente>_reproceso-<hora>_<fecha>`), y como Silver prefiere la versión válida de cada llave, la corrección reemplaza al registro rechazado.

Tracking no se reprocesa por aquí: su origen es la Azure SQL del docente.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if not CATALOGO or not all(c.isalnum() or c == "_" for c in CATALOGO):
    raise ValueError("El catálogo solo admite letras, números y guion bajo")

In [ ]:
dbutils.widgets.dropdown("fuente", "ventas", ["ventas", "devoluciones", "empleados", "resenas", "catalogo", "sucursales"])
dbutils.widgets.text("llaves", "")
FUENTE = dbutils.widgets.get("fuente")
LLAVES = [llave.strip() for llave in dbutils.widgets.get("llaves").split(",") if llave.strip()]

In [ ]:
import os
import sys

RAIZ_BUNDLE = os.path.abspath(os.path.join(os.getcwd(), ".."))
if RAIZ_BUNDLE not in sys.path:
    sys.path.append(RAIZ_BUNDLE)

In [ ]:
import json
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F

from src.electrocasa.utils.preparacion import ORIGEN

CORRECCIONES = {}

filtro = F.col("fuente") == FUENTE
if LLAVES:
    filtro = filtro & F.col("llave").isin(LLAVES)
pendientes = spark.table(f"{CATALOGO}.silver.cuarentena").filter(filtro).select("llave", "reglas_fallidas", "registro")
display(pendientes)

Para corregir, completar `CORRECCIONES` en la celda anterior con la forma `{"llave": {"columna": "valor corregido"}}` y volver a correrla. Los valores van como texto, igual que en el archivo de origen.

In [ ]:
registros = []
for fila in pendientes.collect():
    registro = json.loads(fila["registro"])
    registro.update(CORRECCIONES.get(fila["llave"], {}))
    registros.append({columna: registro.get(columna) for columna in ORIGEN[FUENTE]})

if not registros:
    print("No hay registros en cuarentena para esa fuente y esas llaves")
else:
    ahora = datetime.now()
    sello = f"reproceso-{ahora:%H%M%S}_{ahora:%Y-%m-%d}"
    carpeta = f"/Volumes/{CATALOGO}/default/vol_landing/{FUENTE}"
    if FUENTE == "resenas":
        destino = f"{carpeta}/{FUENTE}_{sello}.json"
        with open(destino, "w", encoding="utf-8") as archivo:
            for registro in registros:
                archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
    elif FUENTE == "catalogo":
        destino = f"{carpeta}/{FUENTE}_{sello}.json"
        with open(destino, "w", encoding="utf-8") as archivo:
            json.dump(registros, archivo, ensure_ascii=False)
    else:
        destino = f"{carpeta}/{FUENTE}_{sello}.csv"
        pd.DataFrame(registros, columns=list(ORIGEN[FUENTE])).to_csv(destino, index=False)
    print(f"Escritos {len(registros)} registros en {destino}")